In [1]:
# Cell 1: imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# For reproducibility and TensorFlow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

RND = 42
tf.random.set_seed(RND)
np.random.seed(RND)

# Cell 2: load dataset (update path if needed)
DATA_PATH = "Churn_Modelling.csv"   # change path if your file is elsewhere
df = pd.read_csv(DATA_PATH)
print("Rows, cols:", df.shape)
df.head()

# Cell 3: inspect columns and select features + target
print(df.columns.tolist())
# Typical columns in this dataset: ['RowNumber','CustomerId','Surname','CreditScore','Geography',
# 'Gender','Age','Tenure','Balance','NumOfProducts','HasCrCard','IsActiveMember','EstimatedSalary','Exited']

# Drop identifier columns that are not predictive
df = df.drop(columns=['RowNumber','CustomerId','Surname'], errors='ignore')

# Target
target_col = 'Exited'   # 1 means left, 0 means stayed

# Cell 4: handle categorical variables
# Convert Gender to binary, Geography to one-hot
df['Gender'] = df['Gender'].fillna('Unknown')
le = LabelEncoder()
df['Gender_enc'] = le.fit_transform(df['Gender'])  # male/female -> 0/1

# One-hot for Geography
geo_dummies = pd.get_dummies(df['Geography'], prefix='Geo', drop_first=True)
df = pd.concat([df.drop(columns=['Geography','Gender']), geo_dummies], axis=1)

# Now define X, y
X = df.drop(columns=[target_col])
y = df[target_col]

print("Feature shape:", X.shape)

# Cell 5: train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RND, stratify=y
)

# Cell 6: normalization (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Optional: show class balance
print("Train class distribution:\n", y_train.value_counts(normalize=True))

# Cell 7: build initial neural network
def build_model(input_dim, hidden_units=[64,32], dropout_rate=0.3, use_batchnorm=True):
    model = Sequential()
    # first layer
    model.add(Dense(hidden_units[0], input_dim=input_dim, activation='relu'))
    if use_batchnorm:
        model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))
    # hidden layers
    for units in hidden_units[1:]:
        model.add(Dense(units, activation='relu'))
        if use_batchnorm:
            model.add(BatchNormalization())
        model.add(Dropout(dropout_rate))
    # output
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

input_dim = X_train_scaled.shape[1]
model = build_model(input_dim, hidden_units=[64,32], dropout_rate=0.25, use_batchnorm=True)
model.summary()

# Cell 8: training with early stopping
es = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1)
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=64,
    callbacks=[es],
    verbose=2
)

# Cell 9: evaluate on test set
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

# Predictions and confusion matrix
y_pred_prob = model.predict(X_test_scaled).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
print("Accuracy (scikit-learn):", acc)
print("\nClassification report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)

# Cell 10: plot confusion matrix
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Stayed','Left'], yticklabels=['Stayed','Left'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# Cell 11: plot training curves (loss & accuracy)
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend(); plt.title("Loss")
plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend(); plt.title("Accuracy")
plt.show()


ModuleNotFoundError: No module named 'tensorflow'

In [4]:
!pip install tensorflow


zsh:1: command not found: pip
